# R2-MJ-14 — immunity against infection, under the gated lead_vPS definition

**Not a Supplementary Results section.** This is a review-response analysis, kept in this chapter so
it runs with the rest of the pipeline and reports through the same mechanism. It writes to
`results/rr14_immunity_infection.json` under `RR14.*` ids.

The approved response to R2-MJ-14 states: 207 genes carry credible sets for both an immune and an
infectious disease; 59 gene–lead-variant pairs carry both on the same lead variant; of the 50 with a
clear direction, 16 (32%) are discordant, against 7.5% genome-wide. The genome-wide figure is now
14.9% because of the sign gate (`chapters/02-analysis-main/README.md`, "Results 4 — lead_vPS and
directional concordance redefined"), while the 32% was computed on 2026-08-18 with no gate. The
letter therefore puts two different rules on either side of one comparison.

Three parts:

1. **Control** — reproduce the 2026-08-18 basis from the refactored tables and assert it, so the
   before column is measured here rather than transcribed.
2. **Recompute** — same class definition, gated direction machinery.
3. **Baseline** — the same gated machinery on a unit the genome-wide figure can also be stated on,
   because 32% is per gene x lead-variant pair on one axis and 14.9% is per lead variant genome-wide.

## Provenance

The 2026-08-18 analysis is
`chapters/_legacy/06-review-r1/directionality-checks/02_immunity_infection_pleiotropy.ipynb`, read
for its class definition and its choice of direction column only. Its logic is reimplemented here
against `data/intermediate_files_refactor`; nothing in `chapters/_legacy/` is executed and none of
its `-r1` CSVs is read.

Every column this analysis needs is byte-identical between `paper.derived("qualifying_credible_sets")`
and the pre-refactor table, and `prioritised_genes_per_cs` reproduces the legacy
`list_of_prioritised_genes_per_CS.parquet` exactly on the qualifying credible sets, so the control
step is a genuine reproduction rather than a re-read of the same file.

## What is held fixed and what changes

**Held fixed.** Immune (`EFO_0000540`) and infectious (`EFO_0005741`) membership from the release's
own **multi-valued** `disease.therapeuticAreas`, which is independent of
`paper.THERAPEUTIC_AREAS` and its legacy ordering — confirmed 2026-08-20. A term carrying both areas
counts as infection. Genes are the L2G-prioritised genes of each credible set. The unit stays
gene x lead variant, and the direction comparison stays on a shared lead variant, because only there
do the two signs refer to the same effect allele.

**Changed — the direction machinery only.** A credible set contributes when its study maps to
**exactly one** disease term and `rescaledStatistics.directionOfEffect` is non-null; where a disease
is carried by more than one contributing study, the most significant by P value gives its direction,
ties broken on `studyLocusId`; direction is the harmonised alternate-allele beta. This is the rule
`chapters/01-data-preparation/07_variant_features.ipynb` writes as the `signedLead*` family, and the
implementation here is asserted against those columns.

The 2026-08-18 exclusion — dropping credible sets mapped to an immune *and* an infection disease at
once — becomes redundant, because such a study carries at least two disease terms. The gate is
strictly stronger: it also drops a study mapped to two immune terms, which the old rule kept.

In [1]:
import itertools

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from scipy import stats

from manuscript_methods import paper

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 400)

numbers = {}

IMMUNE_AREA = "EFO_0000540"  # immune system disease
INFECTION_AREA = "EFO_0005741"  # infectious disease

# Terms named in the response letter or the manuscript, and the three the caveats turn on.
LYMPHOMA = frozenset({"EFO_0000183", "MONDO_0019472"})
PERITONSILLAR_ABSCESS = frozenset({"EFO_0007429"})
COVID = frozenset({"MONDO_0100096"})
TONSILLITIS = "MONDO_0001039"

## Inputs

The multi-valued therapeutic areas come straight from the release's `disease` table; the single-area
hierarchy this pipeline uses elsewhere is deliberately not involved.

In [2]:
CREDIBLE_SET_COLUMNS = [
    "studyId",
    "studyLocusId",
    "variantId",
    "variant",
    "diseaseIds",
    "originalBeta",
    "rescaledStatistics",
    "studyStatistics",
    "variantStatistics",
    "majorLdPopulation",
    "majorLdPopulationAf",
]
credible_sets = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(columns=CREDIBLE_SET_COLUMNS)
    .to_pandas()
)

rescaled = pd.DataFrame(list(credible_sets["rescaledStatistics"]))
credible_sets["direction"] = rescaled["directionOfEffect"].to_numpy()
# The signed effect the 2026-08-18 notebook used: direction x unsigned rescaled beta.
credible_sets["beta"] = credible_sets["direction"] * rescaled["absEstimatedBeta"].to_numpy()
credible_sets["minorAlleleBeta"] = rescaled["minorAlleleEstimatedBeta"].to_numpy()
credible_sets["trait"] = pd.DataFrame(list(credible_sets["studyStatistics"]))["trait"].to_numpy()
variant_statistics = pd.DataFrame(list(credible_sets["variantStatistics"]))
credible_sets["pValueExponent"] = variant_statistics["pValueExponent"].to_numpy()
credible_sets["pValueMantissa"] = variant_statistics["pValueMantissa"].to_numpy()
credible_sets["ldPopulation"] = pd.DataFrame(list(credible_sets["majorLdPopulation"]))["ldPopulation"].to_numpy()
credible_sets["alternateAlleleFrequency"] = pd.DataFrame(list(credible_sets["majorLdPopulationAf"]))[
    "alleleFrequency"
].to_numpy()
credible_sets["effectAllele"] = credible_sets["variant"].apply(lambda v: v["alt"])
credible_sets["otherAllele"] = credible_sets["variant"].apply(lambda v: v["ref"])
credible_sets["diseaseTerms"] = credible_sets["diseaseIds"].apply(lambda ids: 0 if ids is None else len(ids))
credible_sets = credible_sets.drop(
    columns=[
        "variant",
        "rescaledStatistics",
        "studyStatistics",
        "variantStatistics",
        "majorLdPopulation",
        "majorLdPopulationAf",
    ]
)

# The release's own multi-valued therapeutic areas, not paper.THERAPEUTIC_AREAS.
disease = pd.read_parquet(paper.release("disease"), columns=["id", "name", "therapeuticAreas"])
DISEASE_NAME = dict(zip(disease["id"], disease["name"]))
AREAS_OF = {
    identifier: frozenset(areas) if areas is not None else frozenset()
    for identifier, areas in zip(disease["id"], disease["therapeuticAreas"])
}

prioritised = pd.read_parquet(paper.derived("prioritised_genes_per_cs"), columns=["studyLocusId", "geneId"])
target = pd.read_parquet(paper.release("target"), columns=["id", "approvedSymbol"])
SYMBOL = dict(zip(target["id"], target["approvedSymbol"]))

print(f"qualifying credible sets: {len(credible_sets):,}")
print(
    f"with at least one L2G-prioritised gene: "
    f"{prioritised.loc[prioritised['studyLocusId'].isin(set(credible_sets['studyLocusId'])), 'studyLocusId'].nunique():,}"
)
print(
    f"a credible set prioritises up to "
    f"{int(prioritised[prioritised['studyLocusId'].isin(set(credible_sets['studyLocusId']))].groupby('studyLocusId').size().max())} genes"
)

qualifying credible sets: 70,618
with at least one L2G-prioritised gene: 68,675
a credible set prioritises up to 3 genes


## Which column supplied the effect direction on 2026-08-18

The 2026-08-18 notebook formed its signed effect as
`rescaledStatistics.directionOfEffect x rescaledStatistics.absEstimatedBeta`.
`RescaledStatistics.compute_direction_of_effect` in `src/manuscript_methods/rescaled_beta.py` is the
signum of the study's harmonised beta, and `originalBeta` is that beta — the effect of the alternate
allele of `chrom_pos_ref_alt`, fixed across every credible set of a variant. **So the axis analysis
was already on the harmonised alternate-allele direction. It did not use
`rescaledStatistics.minorAlleleEstimatedBeta`**, whose flip is keyed on
`compute_minor_allele_rescaled_beta(major_ancestry_af, ...)` and therefore reverses between studies
of different ancestry whenever the alternate-allele frequency straddles 0.5.

The cell below establishes that from the data rather than from the source, and the one after it
quantifies the counterfactual anyway: how many of the 59 pairs and of the 16 discordant ones could
have been affected, and how many verdicts would have moved.

In [3]:
signed = credible_sets[credible_sets["originalBeta"].notna() & (credible_sets["originalBeta"] != 0)]
agreement = float((np.sign(signed["originalBeta"]) == signed["direction"]).mean())
print(
    f"sign(originalBeta) == directionOfEffect on {agreement:.6f} of {len(signed):,} credible sets with a non-zero beta"
)
assert agreement == 1.0
print(
    f"credible sets with a null direction: {int(credible_sets['direction'].isna().sum()):,} of {len(credible_sets):,}"
)
print(
    f"  of those, with originalBeta present: "
    f"{int((credible_sets['direction'].isna() & credible_sets['originalBeta'].notna()).sum()):,}"
)
print(
    f"credible sets the minor-allele column flips (alternate-allele frequency > 0.5): "
    f"{int((credible_sets['alternateAlleleFrequency'] > 0.5).sum()):,}"
)

sign(originalBeta) == directionOfEffect on 1.000000 of 63,593 credible sets with a non-zero beta
credible sets with a null direction: 7,025 of 70,618
  of those, with originalBeta present: 1
credible sets the minor-allele column flips (alternate-allele frequency > 0.5): 18,889


## Part 1 — control step, the 2026-08-18 basis

The class assignment, the trans-disease exclusion and the majority-sign-over-credible-sets rule of
the 2026-08-18 notebook, reimplemented and asserted against its published counts.

In [4]:
def class_of(disease_id):
    """Immune or infection membership from the release's multi-valued areas; infection wins."""
    areas = AREAS_OF.get(disease_id, frozenset())
    if INFECTION_AREA in areas:
        return "infection"
    if IMMUNE_AREA in areas:
        return "immune"
    return None


def majority_sign(values):
    """Majority sign of a set of effect-allele betas or directions: '+', '-' or 'mixed'."""
    reported = pd.Series(values).dropna()
    positive, negative = int((reported > 0).sum()), int((reported < 0).sum())
    if positive > negative:
        return "+"
    if negative > positive:
        return "-"
    return "mixed"


def verdict_of(immune_sign, infection_sign):
    """Concordant, discordant or undetermined from the two side signs."""
    if "mixed" in (immune_sign, infection_sign):
        return "undetermined"
    return "concordant" if immune_sign == infection_sign else "discordant"


classified = (
    credible_sets.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"}).dropna(subset=["diseaseId"])
)
classified["diseaseName"] = classified["diseaseId"].map(DISEASE_NAME)
classified["class"] = classified["diseaseId"].map(class_of)
classified = classified[classified["class"].notna()]

# A credible set whose study reaches both areas at once puts the identical beta on both sides.
spanning = classified.groupby("studyLocusId")["class"].nunique()
classified["spansBothAreas"] = classified["studyLocusId"].isin(set(spanning[spanning > 1].index))

with_genes = classified.merge(prioritised, on="studyLocusId", how="inner")
with_genes["symbol"] = with_genes["geneId"].map(SYMBOL).fillna(with_genes["geneId"])
comparable = with_genes[~with_genes["spansBothAreas"]]

print(
    f"credible set x disease rows — immune {int((classified['class'] == 'immune').sum()):,}, "
    f"infection {int((classified['class'] == 'infection').sum()):,}"
)
print(
    f"distinct diseases — immune {classified.loc[classified['class'] == 'immune', 'diseaseId'].nunique()}, "
    f"infection {classified.loc[classified['class'] == 'infection', 'diseaseId'].nunique()}"
)
print(f"credible sets reaching both areas at once: {int(spanning.gt(1).sum())}")

credible set x disease rows — immune 9,453, infection 1,143
distinct diseases — immune 107, infection 57
credible sets reaching both areas at once: 81


In [5]:
control_genes = int(
    with_genes.groupby("geneId")["class"].agg(set).map(lambda sides: {"immune", "infection"} <= sides).sum()
)

control_pairs = (
    comparable.groupby(["geneId", "symbol", "variantId", "effectAllele", "otherAllele"])
    .apply(
        lambda group: pd.Series(
            {
                "immune_credible_sets": group.loc[group["class"] == "immune", "studyLocusId"].nunique(),
                "infection_credible_sets": group.loc[group["class"] == "infection", "studyLocusId"].nunique(),
                "immune_sign": majority_sign(group.loc[group["class"] == "immune", "beta"]),
                "infection_sign": majority_sign(group.loc[group["class"] == "infection", "beta"]),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
control_pairs = control_pairs[
    (control_pairs["immune_credible_sets"] > 0) & (control_pairs["infection_credible_sets"] > 0)
].copy()
control_pairs["verdict"] = [
    verdict_of(i, f) for i, f in zip(control_pairs["immune_sign"], control_pairs["infection_sign"])
]
control_determined = control_pairs[control_pairs["verdict"] != "undetermined"]
control_share = round(100 * (control_pairs["verdict"] == "discordant").sum() / len(control_determined), 1)

CONTROL = [
    ("RR14.01", "genes with credible sets in both areas", control_genes, 207),
    ("RR14.02", "gene x lead-variant pairs carrying both", len(control_pairs), 59),
    ("RR14.03", "distinct genes among the pairs", control_pairs["geneId"].nunique(), 44),
    ("RR14.04", "distinct lead variants among the pairs", control_pairs["variantId"].nunique(), 58),
    ("RR14.05", "pairs with a direction on both sides", len(control_determined), 50),
    ("RR14.06", "concordant pairs", int((control_pairs["verdict"] == "concordant").sum()), 34),
    ("RR14.07", "discordant pairs", int((control_pairs["verdict"] == "discordant").sum()), 16),
    ("RR14.08", "undetermined pairs", int((control_pairs["verdict"] == "undetermined").sum()), 9),
    ("RR14.09", "discordant share of determined (%)", control_share, 32.0),
]
print("control against the 2026-08-18 basis")
for identifier, label, got, want in CONTROL:
    numbers[identifier] = got
    print(f"  {identifier}  {label:42s} {got:>6}   2026-08-18 {want:>6}   {'ok' if got == want else 'DIFFERS'}")
    assert got == want, f"{identifier} {label}: {got} != {want}"

control against the 2026-08-18 basis
  RR14.01  genes with credible sets in both areas        207   2026-08-18    207   ok
  RR14.02  gene x lead-variant pairs carrying both        59   2026-08-18     59   ok
  RR14.03  distinct genes among the pairs                 44   2026-08-18     44   ok
  RR14.04  distinct lead variants among the pairs         58   2026-08-18     58   ok
  RR14.05  pairs with a direction on both sides           50   2026-08-18     50   ok
  RR14.06  concordant pairs                               34   2026-08-18     34   ok
  RR14.07  discordant pairs                               16   2026-08-18     16   ok
  RR14.08  undetermined pairs                              9   2026-08-18      9   ok
  RR14.09  discordant share of determined (%)           32.0   2026-08-18   32.0   ok


### Exposure of the control pairs to the minor-allele column

A counterfactual, since the direction column is already harmonised. A pair is exposed when its
signed credible sets span more than one major LD population, or when their alternate-allele
frequencies straddle 0.5 — either way the flip would be applied inconsistently within the pair.

In [6]:
control_keys = set(zip(control_pairs["geneId"], control_pairs["variantId"]))
exposure = []
for (gene_id, variant_id), group in comparable.groupby(["geneId", "variantId"]):
    if (gene_id, variant_id) not in control_keys:
        continue
    with_beta = group[group["beta"].notna()]
    frequencies = with_beta["alternateAlleleFrequency"].dropna()
    exposure.append(
        {
            "symbol": group["symbol"].iloc[0],
            "variantId": variant_id,
            "signed_credible_sets": len(with_beta),
            "ld_populations": ",".join(sorted(set(with_beta["ldPopulation"].dropna()))),
            "n_ld_populations": with_beta["ldPopulation"].nunique(),
            "frequency_straddles_half": bool(
                len(frequencies) and (frequencies > 0.5).any() and (frequencies <= 0.5).any()
            ),
            "any_credible_set_flipped": bool((with_beta["alternateAlleleFrequency"] > 0.5).any()),
            "harmonised_verdict": verdict_of(
                majority_sign(group.loc[group["class"] == "immune", "beta"]),
                majority_sign(group.loc[group["class"] == "infection", "beta"]),
            ),
            "minor_allele_verdict": verdict_of(
                majority_sign(group.loc[group["class"] == "immune", "minorAlleleBeta"]),
                majority_sign(group.loc[group["class"] == "infection", "minorAlleleBeta"]),
            ),
        }
    )
exposure = pd.DataFrame(exposure)
exposed = (exposure["n_ld_populations"] > 1) | exposure["frequency_straddles_half"]
moved = exposure["harmonised_verdict"] != exposure["minor_allele_verdict"]
is_discordant = exposure["harmonised_verdict"] == "discordant"

numbers["RR14.10"] = int(exposed.sum())
numbers["RR14.11"] = int((is_discordant & exposed).sum())
numbers["RR14.12"] = int(moved.sum())
numbers["RR14.13"] = int((is_discordant & moved).sum())

print(f"of the {len(exposure)} control pairs")
print(f"  signed credible sets span >1 major LD population: {int((exposure['n_ld_populations'] > 1).sum())}")
print(f"  alternate-allele frequency straddles 0.5:         {int(exposure['frequency_straddles_half'].sum())}")
print(f"  RR14.10 exposed to an inconsistent flip:          {numbers['RR14.10']}")
print(f"  at least one credible set would be flipped:       {int(exposure['any_credible_set_flipped'].sum())}")
print(f"  RR14.12 verdict would differ:                     {numbers['RR14.12']}")
print(f"\nof the {int(is_discordant.sum())} discordant pairs")
print(f"  RR14.11 exposed:            {numbers['RR14.11']}")
print(f"  RR14.13 verdict would differ: {numbers['RR14.13']}")
print("\nthe pairs whose verdict the column choice moves:")
print(
    exposure[moved][
        [
            "symbol",
            "variantId",
            "signed_credible_sets",
            "ld_populations",
            "frequency_straddles_half",
            "harmonised_verdict",
            "minor_allele_verdict",
        ]
    ].to_string(index=False)
)

of the 59 control pairs
  signed credible sets span >1 major LD population: 39
  alternate-allele frequency straddles 0.5:         2
  RR14.10 exposed to an inconsistent flip:          39
  at least one credible set would be flipped:       22
  RR14.12 verdict would differ:                     2

of the 16 discordant pairs
  RR14.11 exposed:            10
  RR14.13 verdict would differ: 0

the pairs whose verdict the column choice moves:
symbol       variantId  signed_credible_sets ld_populations  frequency_straddles_half harmonised_verdict minor_allele_verdict
 GSDMB 17_39908216_T_C                     2        fin,nfe                      True         concordant           discordant
 EOMES  3_27723132_A_G                     2        eas,nfe                      True         concordant           discordant


## Part 2 — the gate, checked against `variant_features`

Two conditions on the credible set, then one pick per disease. The implementation is asserted against
`signedLeadVPS` and `signedLeadDirectionalConcordance`, so this notebook cannot silently diverge from
the definition `01-data-preparation/07_variant_features.ipynb` writes.

In [7]:
gated = credible_sets[(credible_sets["diseaseTerms"] == 1) & credible_sets["direction"].notna()].copy()
gated["diseaseId"] = gated["diseaseIds"].apply(lambda ids: ids[0])
gated["diseaseName"] = gated["diseaseId"].map(DISEASE_NAME)
gated["class"] = gated["diseaseId"].map(class_of)

print(f"credible sets: {len(credible_sets):,}")
print(f"  study maps to exactly one disease term: {int((credible_sets['diseaseTerms'] == 1).sum()):,}")
print(f"  and directionOfEffect is non-null:      {len(gated):,}   <- contributing")

# The most significant contributing study per disease, ties broken on studyLocusId, as in
# 07_variant_features.ipynb.
gated = gated.sort_values(["pValueExponent", "pValueMantissa", "studyLocusId"])
variant_disease = gated.drop_duplicates(["variantId", "diseaseId"]).copy()
gated_lead_vps = variant_disease.groupby("variantId")["diseaseId"].nunique()

per_variant = variant_disease.groupby("variantId")["direction"].agg(
    diseases="size", up=lambda s: int((s > 0).sum()), down=lambda s: int((s < 0).sum())
)
per_variant["concordance"] = per_variant[["up", "down"]].max(axis=1) / per_variant["diseases"]

print(f"\nvariant x disease rows after the per-disease pick: {len(variant_disease):,}")
print(f"lead variants with >=1 gated disease:  {len(gated_lead_vps):,}")
print(f"lead variants with >=2 gated diseases: {int((gated_lead_vps > 1).sum()):,}")

features = pd.read_parquet(
    paper.derived("variant_features"),
    columns=["variantId", "signedLeadVPS", "signedLeadDirectionalConcordance"],
)
check = features.merge(gated_lead_vps.rename("recomputedVPS"), left_on="variantId", right_index=True, how="left").merge(
    per_variant["concordance"], left_on="variantId", right_index=True, how="left"
)
count_off = check[check["signedLeadVPS"].fillna(0) != check["recomputedVPS"].fillna(0)]
concordance_off = check[
    check["signedLeadDirectionalConcordance"].notna()
    & ((check["signedLeadDirectionalConcordance"] - check["concordance"]).abs() > 1e-9)
]
print(f"\nlead variants in variant_features: {len(features):,}")
print(f"  gated disease count differs from signedLeadVPS:                 {len(count_off)}")
print(f"  concordance differs from signedLeadDirectionalConcordance:      {len(concordance_off)}")
assert len(count_off) == 0
assert len(concordance_off) == 0

credible sets: 70,618
  study maps to exactly one disease term: 65,431
  and directionOfEffect is non-null:      59,725   <- contributing



variant x disease rows after the per-disease pick: 46,816
lead variants with >=1 gated disease:  35,472
lead variants with >=2 gated diseases: 5,919

lead variants in variant_features: 40,706
  gated disease count differs from signedLeadVPS:                 0
  concordance differs from signedLeadDirectionalConcordance:      0


## Part 3 — the gated axis

Each side's direction is now the majority over its gated **diseases** rather than over credible sets,
because the per-disease pick leaves one signed direction per disease — the same unit
`signedLeadDirectionalConcordance` is computed over.

In [8]:
axis = gated[gated["class"].notna()].copy()
dual_area = sorted(
    {d for d in axis["diseaseId"].unique() if {IMMUNE_AREA, INFECTION_AREA} <= AREAS_OF.get(d, frozenset())}
)
print(
    f"gated credible sets on the axis — immune {int((axis['class'] == 'immune').sum()):,}, "
    f"infection {int((axis['class'] == 'infection').sum()):,}"
)
print(
    f"gated diseases — immune {axis.loc[axis['class'] == 'immune', 'diseaseId'].nunique()}, "
    f"infection {axis.loc[axis['class'] == 'infection', 'diseaseId'].nunique()}"
)
print(f"terms carrying both areas, counted as infection: {[f'{d} ({DISEASE_NAME.get(d)})' for d in dual_area]}")

axis_genes = axis.merge(prioritised, on="studyLocusId", how="inner")
axis_genes["symbol"] = axis_genes["geneId"].map(SYMBOL).fillna(axis_genes["geneId"])
# The per-disease pick inside the unit of analysis: one row per gene x lead variant x disease.
axis_pick = axis_genes.sort_values(["pValueExponent", "pValueMantissa", "studyLocusId"]).drop_duplicates(
    ["geneId", "variantId", "diseaseId"]
)
gated_gene_ids = {
    gene for gene, sides in axis_pick.groupby("geneId")["class"].agg(set).items() if {"immune", "infection"} <= sides
}

rows = []
for (gene_id, variant_id), group in axis_pick.groupby(["geneId", "variantId"]):
    immune, infection = group[group["class"] == "immune"], group[group["class"] == "infection"]
    if not len(immune) or not len(infection):
        continue
    immune_sign, infection_sign = majority_sign(immune["direction"]), majority_sign(infection["direction"])
    rows.append(
        {
            "geneId": gene_id,
            "symbol": group["symbol"].iloc[0],
            "variantId": variant_id,
            "effectAllele": group["effectAllele"].iloc[0],
            "otherAllele": group["otherAllele"].iloc[0],
            "gated_lead_vps": int(gated_lead_vps.get(variant_id, 0)),
            "gated_concordance": round(float(per_variant.loc[variant_id, "concordance"]), 4),
            "immune_diseases": len(immune),
            "infection_diseases": len(infection),
            "immune_sign": immune_sign,
            "infection_sign": infection_sign,
            "immune_median_beta": round(float(immune["beta"].median()), 4),
            "infection_median_beta": round(float(infection["beta"].median()), 4),
            "verdict": verdict_of(immune_sign, infection_sign),
            "immune_disease_names": "; ".join(sorted(set(immune["diseaseName"].dropna()))),
            "infection_disease_names": "; ".join(sorted(set(infection["diseaseName"].dropna()))),
        }
    )
gated_pairs = pd.DataFrame(rows).sort_values(["immune_diseases", "infection_diseases"], ascending=False)
gated_determined = gated_pairs[gated_pairs["verdict"] != "undetermined"]
gated_share = round(100 * (gated_pairs["verdict"] == "discordant").sum() / len(gated_determined), 1)

numbers["RR14.14"] = len(gated_gene_ids)
numbers["RR14.15"] = len(gated_pairs)
numbers["RR14.16"] = gated_pairs["geneId"].nunique()
numbers["RR14.17"] = gated_pairs["variantId"].nunique()
numbers["RR14.18"] = len(gated_determined)
numbers["RR14.19"] = int((gated_pairs["verdict"] == "concordant").sum())
numbers["RR14.20"] = int((gated_pairs["verdict"] == "discordant").sum())
numbers["RR14.21"] = int((gated_pairs["verdict"] == "undetermined").sum())
numbers["RR14.22"] = gated_share

comparison = pd.DataFrame(
    [
        ("genes with credible sets in both areas", "RR14.01", "RR14.14"),
        ("gene x lead-variant pairs carrying both", "RR14.02", "RR14.15"),
        ("  distinct genes", "RR14.03", "RR14.16"),
        ("  distinct lead variants", "RR14.04", "RR14.17"),
        ("pairs with a direction on both sides", "RR14.05", "RR14.18"),
        ("  concordant", "RR14.06", "RR14.19"),
        ("  discordant", "RR14.07", "RR14.20"),
        ("  undetermined", "RR14.08", "RR14.21"),
        ("discordant share of determined (%)", "RR14.09", "RR14.22"),
    ],
    columns=["quantity", "before_id", "after_id"],
)
comparison["before"] = comparison["before_id"].map(numbers)
comparison["after"] = comparison["after_id"].map(numbers)
print(comparison[["quantity", "before_id", "before", "after_id", "after"]].to_string(index=False))

gated credible sets on the axis — immune 7,497, infection 667
gated diseases — immune 93, infection 46
terms carrying both areas, counted as infection: ['EFO_0007429 (peritonsillar abscess)']


                               quantity before_id  before after_id  after
 genes with credible sets in both areas   RR14.01   207.0  RR14.14  115.0
gene x lead-variant pairs carrying both   RR14.02    59.0  RR14.15   34.0
                         distinct genes   RR14.03    44.0  RR14.16   28.0
                 distinct lead variants   RR14.04    58.0  RR14.17   34.0
   pairs with a direction on both sides   RR14.05    50.0  RR14.18   32.0
                             concordant   RR14.06    34.0  RR14.19   21.0
                             discordant   RR14.07    16.0  RR14.20   11.0
                           undetermined   RR14.08     9.0  RR14.21    2.0
     discordant share of determined (%)   RR14.09    32.0  RR14.22   34.4


In [9]:
print("every gated gene x lead-variant pair, most gated immune diseases first:")
print(
    gated_pairs[
        [
            "symbol",
            "variantId",
            "effectAllele",
            "gated_lead_vps",
            "gated_concordance",
            "immune_diseases",
            "infection_diseases",
            "immune_sign",
            "infection_sign",
            "immune_median_beta",
            "infection_median_beta",
            "verdict",
            "infection_disease_names",
        ]
    ].to_string(index=False)
)

every gated gene x lead-variant pair, most gated immune diseases first:
         symbol        variantId effectAllele  gated_lead_vps  gated_concordance  immune_diseases  infection_diseases immune_sign infection_sign  immune_median_beta  infection_median_beta      verdict                                                                                                                         infection_disease_names
          SH2B3 12_111446804_T_C            C              29             0.8966                9                   2           -              -             -0.1109                -0.0334   concordant                                                                            Prosthesis-Related Infections; respiratory tract infectious disorder
            FLG  1_152313385_G_A            A              13             0.9231                3                   2           +              +              0.5458                 0.1298   concordant                                      

### Which pairs drop, and which side lost its direction

For each of the 59 control pairs: the credible sets each side had, how many survive the
single-disease-study condition, how many survive the sign gate on top of it, and the verdict before
and after. A pair drops when a side reaches zero.

In [10]:
gated_verdict = dict(zip(zip(gated_pairs["geneId"], gated_pairs["variantId"]), gated_pairs["verdict"]))
control_verdict = dict(zip(zip(control_pairs["geneId"], control_pairs["variantId"]), control_pairs["verdict"]))

fate = []
for (gene_id, variant_id), group in comparable.groupby(["geneId", "variantId"]):
    if (gene_id, variant_id) not in control_keys:
        continue
    record = {
        "symbol": group["symbol"].iloc[0],
        "geneId": gene_id,
        "variantId": variant_id,
        "before": control_verdict[(gene_id, variant_id)],
        "after": gated_verdict.get((gene_id, variant_id), "dropped"),
    }
    reasons = []
    for side in ("immune", "infection"):
        side_rows = group[group["class"] == side]
        single = side_rows[side_rows["diseaseTerms"] == 1]
        record[f"{side}_credible_sets"] = side_rows["studyLocusId"].nunique()
        record[f"{side}_single_disease"] = single["studyLocusId"].nunique()
        record[f"{side}_signed"] = single[single["direction"].notna()]["studyLocusId"].nunique()
        if record["after"] == "dropped" and record[f"{side}_signed"] == 0:
            if record[f"{side}_single_disease"] == 0:
                reasons.append(
                    f"{side}: no qualifying single-disease study "
                    f"({record[f'{side}_credible_sets']} credible sets, every study multi-term)"
                )
            else:
                reasons.append(
                    f"{side}: no signed effect ({record[f'{side}_single_disease']} single-disease "
                    f"credible sets, none carrying a direction)"
                )
    record["reason"] = "; ".join(reasons)
    fate.append(record)
fate = pd.DataFrame(fate)
dropped = fate[fate["after"] == "dropped"]

numbers["RR14.23"] = len(dropped)
numbers["RR14.24"] = len(gated_pairs) - int((fate["after"] != "dropped").sum())
numbers["RR14.25"] = int((dropped["infection_signed"] == 0).sum())
numbers["RR14.26"] = int((dropped["immune_signed"] == 0).sum())

print("before x after:")
print(pd.crosstab(fate["before"], fate["after"]).to_string())
print(f"\nRR14.23 pairs dropped: {numbers['RR14.23']}")
print(f"RR14.24 pairs the gate adds, absent from the 59: {numbers['RR14.24']}")
print(f"RR14.25 dropped pairs losing the infection side: {numbers['RR14.25']}")
print(f"RR14.26 dropped pairs losing the immune side:    {numbers['RR14.26']}")
print(f"        losing both: {int(((dropped['infection_signed'] == 0) & (dropped['immune_signed'] == 0)).sum())}")
for side in ("immune", "infection"):
    lost = dropped[dropped[f"{side}_signed"] == 0]
    print(
        f"        {side}: no qualifying single-disease study "
        f"{int((lost[f'{side}_single_disease'] == 0).sum())}, "
        f"single-disease studies present but none signed "
        f"{int((lost[f'{side}_single_disease'] > 0).sum())}"
    )

print("\nthe dropped pairs:")
print(
    dropped[
        [
            "symbol",
            "variantId",
            "before",
            "immune_credible_sets",
            "immune_single_disease",
            "immune_signed",
            "infection_credible_sets",
            "infection_single_disease",
            "infection_signed",
            "reason",
        ]
    ].to_string(index=False)
)
print("\npairs that survive with a different verdict:")
print(
    fate[(fate["after"] != "dropped") & (fate["after"] != fate["before"])][
        [
            "symbol",
            "variantId",
            "before",
            "after",
            "immune_credible_sets",
            "immune_signed",
            "infection_credible_sets",
            "infection_signed",
        ]
    ].to_string(index=False)
)

before x after:
after         concordant  discordant  dropped  undetermined
before                                                     
concordant            21           0       12             1
discordant             0          10        6             0
undetermined           0           1        7             1

RR14.23 pairs dropped: 25
RR14.24 pairs the gate adds, absent from the 59: 0
RR14.25 dropped pairs losing the infection side: 21
RR14.26 dropped pairs losing the immune side:    5
        losing both: 1
        immune: no qualifying single-disease study 4, single-disease studies present but none signed 1
        infection: no qualifying single-disease study 17, single-disease studies present but none signed 4

the dropped pairs:
 symbol        variantId       before  immune_credible_sets  immune_single_disease  immune_signed  infection_credible_sets  infection_single_disease  infection_signed                                                                                    

## Part 4 — a like-for-like baseline

32% is per gene x lead-variant pair on one axis; 14.9% is per lead variant genome-wide, and it is
reported over **cluster representatives**, while this analysis is not restricted to representatives.
Neither the old comparison nor "the new share against 14.9%" is like for like. Two units, same gated
machinery:

**Per lead variant**, over every lead variant with at least two gated diseases — the same universe
the axis analysis draws on.

**Per disease pair on one gated lead variant** — every unordered pair of a variant's gated diseases.
The cross-area version keeps the pairs whose two terms share no therapeutic area at all, which is the
closest structural match to an immune-against-infection pair, and the axis version keeps one immune
term against one infection term.

The per-variant unit is not comparable across the two, because the chance of carrying at least one
opposing direction rises with how many diseases a variant carries. The per-pair unit removes most of
that and a lead_vPS-bin-matched baseline removes the rest.

In [11]:
pleiotropic = per_variant[per_variant["diseases"] > 1]
numbers["RR14.27"] = len(pleiotropic)
numbers["RR14.28"] = int((pleiotropic["concordance"] < 1).sum())
numbers["RR14.29"] = round(100 * float((pleiotropic["concordance"] < 1).mean()), 1)

print("--- per lead variant, gated ---")
print(f"RR14.27 lead variants with >=2 gated diseases: {numbers['RR14.27']:,}")
print(f"        fully concordant: {int((pleiotropic['concordance'] == 1).sum()):,}")
print(f"RR14.28 at least one opposing direction: {numbers['RR14.28']:,}")
print(f"RR14.29 that share: {numbers['RR14.29']}%")
print("        the letter's 14.9% (322 of 2,166) is this quantity over cluster representatives only")

axis_variants = set(gated_pairs["variantId"])
selected = pleiotropic.loc[[v for v in axis_variants if v in pleiotropic.index]]
print(
    f"\nthe {len(selected)} lead variants carrying a gated pair: "
    f"{int((selected['concordance'] < 1).sum())}/{len(selected)} = "
    f"{(selected['concordance'] < 1).mean():.1%} — a pleiotropy artefact, see below"
)

--- per lead variant, gated ---
RR14.27 lead variants with >=2 gated diseases: 5,919
        fully concordant: 4,868
RR14.28 at least one opposing direction: 1,051
RR14.29 that share: 17.8%
        the letter's 14.9% (322 of 2,166) is this quantity over cluster representatives only

the 34 lead variants carrying a gated pair: 22/34 = 64.7% — a pleiotropy artefact, see below


In [12]:
records = []
for variant_id, group in variant_disease.groupby("variantId"):
    items = sorted(zip(group["diseaseId"], group["direction"]))
    if len(items) < 2:
        continue
    for (first, first_sign), (second, second_sign) in itertools.combinations(items, 2):
        first_areas, second_areas = AREAS_OF.get(first, frozenset()), AREAS_OF.get(second, frozenset())
        records.append(
            (
                variant_id,
                first,
                second,
                first_sign == second_sign,
                len(first_areas & second_areas) == 0,
                frozenset({class_of(first), class_of(second)}) == frozenset({"immune", "infection"}),
            )
        )
disease_pairs = pd.DataFrame(
    records,
    columns=["variantId", "firstDisease", "secondDisease", "same_direction", "different_areas", "on_axis"],
)
disease_pairs["gated_lead_vps"] = disease_pairs["variantId"].map(gated_lead_vps)

axis_pairs = disease_pairs[disease_pairs["on_axis"]]
cross_area = disease_pairs[disease_pairs["different_areas"]]

numbers["RR14.30"] = len(disease_pairs)
numbers["RR14.31"] = round(100 * float((~disease_pairs["same_direction"]).mean()), 1)
numbers["RR14.32"] = len(cross_area)
numbers["RR14.33"] = round(100 * float((~cross_area["same_direction"]).mean()), 1)
numbers["RR14.34"] = len(axis_pairs)
numbers["RR14.35"] = round(100 * float((~axis_pairs["same_direction"]).mean()), 1)

print("--- per disease pair on one gated lead variant ---")
for label, frame in [
    ("all disease pairs", disease_pairs),
    ("pairs whose two terms share no therapeutic area", cross_area),
    ("the axis: one immune term against one infection term", axis_pairs),
]:
    print(
        f"{label:54s} {len(frame):>7,} pairs on {frame['variantId'].nunique():>6,} lead variants   "
        f"discordant {int((~frame['same_direction']).sum()):>6,} = {(~frame['same_direction']).mean():.1%}"
    )
for label, wanted in [("both terms immune-area", {"immune"}), ("both terms infection-area", {"infection"})]:
    mask = [
        frozenset({class_of(a), class_of(b)}) == frozenset(wanted)
        for a, b in zip(disease_pairs["firstDisease"], disease_pairs["secondDisease"])
    ]
    frame = disease_pairs[mask]
    print(
        f"{label:54s} {len(frame):>7,} pairs on {frame['variantId'].nunique():>6,} lead variants   "
        f"discordant {int((~frame['same_direction']).sum()):>6,} = {(~frame['same_direction']).mean():.1%}"
    )

odds_ratio, p_value = stats.fisher_exact(
    [
        [int((~axis_pairs["same_direction"]).sum()), int(axis_pairs["same_direction"].sum())],
        [int((~cross_area["same_direction"]).sum()), int(cross_area["same_direction"].sum())],
    ]
)
numbers["RR14.36"] = round(float(p_value), 3)
print(f"\naxis against the cross-area baseline: OR = {odds_ratio:.2f}, RR14.36 P = {numbers['RR14.36']}")

--- per disease pair on one gated lead variant ---
all disease pairs                                       35,997 pairs on  5,919 lead variants   discordant  7,222 = 20.1%
pairs whose two terms share no therapeutic area         19,147 pairs on  2,636 lead variants   discordant  5,099 = 26.6%
the axis: one immune term against one infection term        97 pairs on     38 lead variants   discordant     24 = 24.7%
both terms immune-area                                   2,010 pairs on    745 lead variants   discordant    266 = 13.2%
both terms infection-area                                   42 pairs on     22 lead variants   discordant      5 = 11.9%

axis against the cross-area baseline: OR = 0.91, RR14.36 P = 0.731


In [13]:
# Per-pair discordance itself rises with the variant's pleiotropy, so the baseline is reweighted to
# the axis pairs' distribution of gated lead_vPS.
EDGES = [2, 3, 5, 10, 20, 10_000]
LABELS = ["2", "3-4", "5-9", "10-19", ">=20"]
for name, frame in [("cross-area baseline", cross_area), ("the axis", axis_pairs)]:
    binned = frame.assign(bin=pd.cut(frame["gated_lead_vps"], EDGES, right=False, labels=LABELS))
    summary = binned.groupby("bin", observed=True).agg(
        pairs=("same_direction", "size"), discordant=("same_direction", lambda s: int((~s).sum()))
    )
    summary["share"] = (summary["discordant"] / summary["pairs"]).map("{:.1%}".format)
    print(f"{name}, by the variant's gated lead_vPS:")
    print(summary.to_string(), "\n")

weights = pd.cut(axis_pairs["gated_lead_vps"], EDGES, right=False, labels=LABELS).value_counts(normalize=True)
binned_baseline = cross_area.assign(bin=pd.cut(cross_area["gated_lead_vps"], EDGES, right=False, labels=LABELS))
matched = sum(
    weight * float((~binned_baseline.loc[binned_baseline["bin"] == label, "same_direction"]).mean())
    for label, weight in weights.items()
    if (binned_baseline["bin"] == label).any()
)
numbers["RR14.37"] = round(100 * matched, 1)
print(f"RR14.37 lead_vPS-bin-matched cross-area baseline: {numbers['RR14.37']}%")
print(f"RR14.35 the axis on the same unit:                {numbers['RR14.35']}%")

axis_variant_vps = gated_lead_vps[list(axis_pairs["variantId"].unique())]
print(
    f"\ngated lead_vPS of the {len(axis_variant_vps)} lead variants carrying an axis disease pair: "
    f"median {float(axis_variant_vps.median()):.1f}, mean {float(axis_variant_vps.mean()):.1f}"
)
print(
    f"gated lead_vPS of all {len(pleiotropic):,} lead variants with >=2 gated diseases: "
    f"median {float(pleiotropic['diseases'].median()):.1f}, mean {float(pleiotropic['diseases'].mean()):.1f}"
)

cross-area baseline, by the variant's gated lead_vPS:
       pairs  discordant  share
bin                            
2       1248         288  23.1%
3-4     2315         484  20.9%
5-9     3285         810  24.7%
10-19   3720         952  25.6%
>=20    8579        2565  29.9% 

the axis, by the variant's gated lead_vPS:
       pairs  discordant  share
bin                            
2          7           2  28.6%
3-4       16           7  43.8%
5-9       21          11  52.4%
10-19     19           3  15.8%
>=20      34           1   2.9% 

RR14.37 lead_vPS-bin-matched cross-area baseline: 25.9%
RR14.35 the axis on the same unit:                24.7%

gated lead_vPS of the 38 lead variants carrying an axis disease pair: median 5.0, mean 10.8
gated lead_vPS of all 5,919 lead variants with >=2 gated diseases: median 2.0, mean 2.9


### Sensitivity to the two ontology caveats and to COVID-19

Three infection-side terms are worth removing one at a time: the lymphoma terms, which take the
infectious-disease area from their viral aetiology and are cancers; peritonsillar abscess, the one
corpus term carrying **both** areas and assigned to infection by the tie-break; and COVID-19, which
dominates the side.

In [14]:
def rebuild(excluded):
    """The pair-level and disease-pair-level figures with some infection terms removed."""
    frame = axis_pick[~((axis_pick["class"] == "infection") & axis_pick["diseaseId"].isin(excluded))]
    verdicts = []
    for _, group in frame.groupby(["geneId", "variantId"]):
        immune, infection = group[group["class"] == "immune"], group[group["class"] == "infection"]
        if not len(immune) or not len(infection):
            continue
        verdicts.append(verdict_of(majority_sign(immune["direction"]), majority_sign(infection["direction"])))
    verdicts = pd.Series(verdicts, dtype=object)
    determined = int((verdicts != "undetermined").sum())
    discordant = int((verdicts == "discordant").sum())
    kept = axis_pairs[~(axis_pairs["firstDisease"].isin(excluded) | axis_pairs["secondDisease"].isin(excluded))]
    return len(verdicts), determined, discordant, len(kept), int((~kept["same_direction"]).sum())


SCENARIOS = [
    ("as computed", frozenset()),
    ("excluding the lymphoma terms", LYMPHOMA),
    ("excluding peritonsillar abscess", PERITONSILLAR_ABSCESS),
    ("excluding both", LYMPHOMA | PERITONSILLAR_ABSCESS),
    ("excluding COVID-19", COVID),
    ("excluding all three", LYMPHOMA | PERITONSILLAR_ABSCESS | COVID),
]
sensitivity = []
for label, excluded in SCENARIOS:
    pairs_n, determined_n, discordant_n, dp_n, dp_discordant = rebuild(excluded)
    sensitivity.append(
        {
            "removed": label,
            "pairs": pairs_n,
            "determined": determined_n,
            "discordant": discordant_n,
            "pair share (%)": round(100 * discordant_n / max(determined_n, 1), 1),
            "disease pairs": dp_n,
            "dp discordant": dp_discordant,
            "disease-pair share (%)": round(100 * dp_discordant / max(dp_n, 1), 1),
        }
    )
sensitivity = pd.DataFrame(sensitivity)
numbers["RR14.38"] = float(sensitivity.loc[sensitivity["removed"] == "excluding COVID-19", "pair share (%)"].iloc[0])
numbers["RR14.39"] = float(
    sensitivity.loc[sensitivity["removed"] == "excluding COVID-19", "disease-pair share (%)"].iloc[0]
)
print(sensitivity.to_string(index=False))

                        removed  pairs  determined  discordant  pair share (%)  disease pairs  dp discordant  disease-pair share (%)
                    as computed     34          32          11            34.4             97             24                    24.7
   excluding the lymphoma terms     30          28          10            35.7             90             22                    24.4
excluding peritonsillar abscess     28          28          10            35.7             82             19                    23.2
                 excluding both     24          24           9            37.5             75             17                    22.7
             excluding COVID-19     26          25           5            20.0             84             13                    15.5
            excluding all three     15          15           2            13.3             62              6                     9.7


## Part 5 — the named examples

Each is named in the response letter or in the manuscript, so any change in its direction, its
concordance or its single-credible-set support has to be flagged. TYK2 P1104A is carried along
because the 2026-08-18 write-up uses it as the coverage-limit case.

In [15]:
NAMED = [
    ("IL23R / C1orf141", "1_67131436_G_A", "RR14.40", "RR14.41"),
    ("CCDC88B", "11_64340263_G_A", "RR14.42", "RR14.43"),
    ("TYK2", "19_10355447_C_T", "RR14.44", "RR14.45"),
    ("LACC1", "13_43883789_A_G", "RR14.46", "RR14.47"),
    ("TYK2 P1104A", "19_10352442_G_C", "RR14.48", "RR14.49"),
]

for label, variant_id, vps_id, concordance_id in NAMED:
    diseases = variant_disease[variant_disease["variantId"] == variant_id]
    all_credible_sets = credible_sets[credible_sets["variantId"] == variant_id]
    numbers[vps_id] = len(diseases)
    numbers[concordance_id] = round(float(per_variant.loc[variant_id, "concordance"]), 4)
    up, down = int((diseases["direction"] > 0).sum()), int((diseases["direction"] < 0).sum())
    print("=" * 118)
    print(f"{label}  {variant_id}   effect allele {all_credible_sets['effectAllele'].iloc[0]}")
    print(
        f"  {vps_id} gated lead_vPS {numbers[vps_id]} ({up} up / {down} down)   "
        f"{concordance_id} concordance {numbers[concordance_id]:.4f}"
    )
    print(
        f"  credible sets on the variant {len(all_credible_sets)}; dropped as multi-term "
        f"{int((all_credible_sets['diseaseTerms'] > 1).sum())}; dropped as unsigned "
        f"{int(all_credible_sets['direction'].isna().sum())}"
    )
    pair = gated_pairs[gated_pairs["variantId"] == variant_id]
    for _, row in pair.iterrows():
        print(
            f"  {row['symbol']}: immune {row['immune_sign']} ({row['immune_disease_names']}) | "
            f"infection {row['infection_sign']} ({row['infection_disease_names']}) -> {row['verdict']}"
        )
    if not len(pair):
        sides = sorted(set(axis_pick.loc[axis_pick["variantId"] == variant_id, "class"]))
        print(f"  no gated pair: areas present on the gated credible sets = {sides or 'none'}")
    print("  every gated disease, by effect:")
    for row in diseases.sort_values("beta").itertuples():
        print(f"    {row.beta:+.4f}  {row.diseaseName}  ({class_of(row.diseaseId) or 'neither area'})")

IL23R / C1orf141  1_67131436_G_A   effect allele A
  RR14.40 gated lead_vPS 2 (1 up / 1 down)   RR14.41 concordance 0.5000
  credible sets on the variant 2; dropped as multi-term 0; dropped as unsigned 0
  C1orf141: immune - (Crohn's disease) | infection + (leprosy) -> discordant
  every gated disease, by effect:
    -0.2739  Crohn's disease  (immune)
    +0.6970  leprosy  (infection)
CCDC88B  11_64340263_G_A   effect allele A
  RR14.42 gated lead_vPS 8 (3 up / 5 down)   RR14.43 concordance 0.6250
  credible sets on the variant 14; dropped as multi-term 0; dropped as unsigned 2
  CCDC88B: immune - (autoimmune disease; immune system disease; psoriasis) | infection + (leprosy) -> discordant
  every gated disease, by effect:
    -0.1807  sclerosing cholangitis  (neither area)
    -0.1471  sarcoidosis  (neither area)
    -0.1289  immune system disease  (immune)
    -0.0697  psoriasis  (immune)
    -0.0395  autoimmune disease  (immune)
    +0.0542  basal cell carcinoma  (neither area)
    +

## Part 6 — do the known caveats survive gating

COVID-19's share of the infection side, the lymphoma terms, and whether any trans-disease study can
still supply both sides of a pair. The infection side is counted as distinct (lead variant, disease)
rows on the lead variants carrying a gated pair, so a credible set prioritising two genes is not
counted twice.

In [16]:
infection_side = axis_pick[
    (axis_pick["class"] == "infection") & axis_pick["variantId"].isin(axis_variants)
].drop_duplicates(["variantId", "diseaseId"])
profile = (
    infection_side.groupby(["diseaseId", "diseaseName"])
    .agg(gated_rows=("variantId", "size"), lead_variants=("variantId", "nunique"))
    .reset_index()
    .sort_values("gated_rows", ascending=False)
)
print("the infection side of the surviving pairs:")
print(profile.to_string(index=False))

numbers["RR14.50"] = len(infection_side)
numbers["RR14.51"] = int(infection_side["diseaseId"].isin(COVID).sum())
numbers["RR14.52"] = int(infection_side["diseaseId"].isin(LYMPHOMA).sum())
numbers["RR14.53"] = int(infection_side["diseaseId"].isin(PERITONSILLAR_ABSCESS).sum())
numbers["RR14.54"] = int((axis_pick["diseaseTerms"] > 1).sum())

covid = infection_side[infection_side["diseaseId"].isin(COVID)]
lymphoma = infection_side[infection_side["diseaseId"].isin(LYMPHOMA)]
print(f"\nRR14.50 gated infection-side (lead variant, disease) rows: {numbers['RR14.50']}")
print(
    f"RR14.51 COVID-19: {numbers['RR14.51']} of them "
    f"({numbers['RR14.51'] / numbers['RR14.50']:.1%}) on {covid['variantId'].nunique()} lead variants"
)
print("        was 49 comparable credible sets on 16 lead variants")
print(
    f"RR14.52 lymphoma terms: {numbers['RR14.52']} rows on {lymphoma['variantId'].nunique()} "
    f"lead variants — {sorted(set(lymphoma['diseaseName']))}"
)
print(
    f"RR14.53 peritonsillar abscess: {numbers['RR14.53']} rows — the only corpus term carrying both "
    f"areas, assigned to infection by the tie-break"
)
print(
    f"RR14.54 trans-disease credible sets supplying both sides of a pair: {numbers['RR14.54']} — "
    f"zero by construction, the gate requires exactly one disease term"
)

for term in dual_area:
    print(f"\n{term} ({DISEASE_NAME.get(term)}) areas: {sorted(AREAS_OF[term])}")
print(f"\n{TONSILLITIS} ({DISEASE_NAME.get(TONSILLITIS)}) areas: {sorted(AREAS_OF.get(TONSILLITIS, frozenset()))}")
print("  it carries the immune-system area and NOT the infectious-disease area, so under this class")
print("  definition tonsillitis sits on the immune side and is not an infection counter-signal here")
assert IMMUNE_AREA in AREAS_OF[TONSILLITIS] and INFECTION_AREA not in AREAS_OF[TONSILLITIS]

the infection side of the surviving pairs:
    diseaseId                                diseaseName  gated_rows  lead_variants
MONDO_0100096                                   COVID-19           9              9
  EFO_0007429                      peritonsillar abscess           7              7
MONDO_0001628                              tinea unguium           6              6
MONDO_0004678                            dermatophytosis           5              5
  EFO_0001054                                    leprosy           4              4
  EFO_0000183                          Hodgkins lymphoma           3              3
MONDO_0024355      respiratory tract infectious disorder           3              3
  EFO_0007512                                tinea pedis           2              2
  EFO_1001406              Prosthesis-Related Infections           2              2
  EFO_0000771                          bacterial disease           1              1
  EFO_0005741                    

## Write the results

In [17]:
print(paper.save_results("rr14_immunity_infection", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/rr14_immunity_infection.json


,computed
RR14.01,207.0000
RR14.02,59.0000
RR14.03,44.0000
RR14.04,58.0000
RR14.05,50.0000
RR14.06,34.0000
RR14.07,16.0000
RR14.08,9.0000
RR14.09,32.0000
RR14.10,39.0000


# What this shows

## The old basis reproduces, and its direction column was never the problem

Every one of the nine 2026-08-18 counts reproduces from `data/intermediate_files_refactor` and is
asserted: 207 genes, 59 pairs over 44 genes and 58 lead variants, 50 determined, 34 / 16 / 9, 32.0%.

That analysis signed its effects with `rescaledStatistics.directionOfEffect x absEstimatedBeta`, and
`directionOfEffect` is the signum of `originalBeta` — the harmonised alternate-allele effect — on
63,593 of 63,593 credible sets carrying a non-zero beta. **It did not use
`minorAlleleEstimatedBeta`**, so the axis figure was never exposed to that column's ancestry-keyed
flip.

The counterfactual was quantified anyway, because 18,889 credible sets corpus-wide would be flipped.
Of the 59 pairs, **39 are exposed** (39 span more than one major LD population, 2 straddle an
alternate-allele frequency of 0.5) and 22 contain at least one credible set the flip reverses. Had
the minor-allele column been used, **2 verdicts would have moved** — GSDMB `17_39908216_T_C` and
EOMES `3_27723132_A_G`, both concordant to discordant. Of the 16 discordant pairs, 10 are exposed and
**none** would change. The 32% is not a minor-allele artefact in either direction.

## The gate is a coverage cut, not a direction cut

The gate is checked against the pipeline rather than re-derived: the per-variant gated disease count
and concordance agree with `variant_features.signedLeadVPS` and
`signedLeadDirectionalConcordance` on all 40,706 lead variants, 0 disagreements either way.

| quantity | before | after |
| --- | --- | --- |
| genes with credible sets in both areas | 207 | **115** |
| gene x lead-variant pairs carrying both | 59 | **34** |
| — distinct genes | 44 | **28** |
| — distinct lead variants | 58 | **34** |
| pairs with a direction on both sides | 50 | **32** |
| — concordant | 34 | **21** |
| — discordant | 16 | **11** |
| — undetermined | 9 | **2** |
| discordant share of determined | 32.0% | **34.4%** |

**25 pairs drop and the gate adds none**, so the 34 are a subset of the 59. 21 lose the infection
side, 5 lose the immune side, and FTO `16_53787213_A_G` loses both. By cause — infection side: 17
because every credible set there comes from a multi-term study, 4 because the surviving
single-disease credible sets carry no signed effect; immune side: 4 and 1.

| before \ after | concordant | discordant | dropped | undetermined |
| --- | --- | --- | --- | --- |
| concordant | 21 | 0 | 12 | 1 |
| discordant | 0 | 10 | 6 | 0 |
| undetermined | 0 | 1 | 7 | 1 |

Only two pairs survive with a different verdict — FUT2 `19_48700572_C_T` undetermined to discordant,
`ENSG00000293584` `2_111429464_A_G` concordant to undetermined. **No concordant pair becomes
discordant.** The undetermined group nearly vanishes (9 to 2) because the gate leaves exactly one
signed direction per disease, so "mixed" now needs a genuine tie in disease counts rather than a
missing beta.

## On a matched unit the axis stops standing out

| unit, all gated | count | discordant |
| --- | --- | --- |
| lead variants with 2 or more gated diseases | 5,919 | 1,051 = **17.8%** |
| — the letter's 14.9% is this quantity over cluster representatives only (322 of 2,166) | | |
| all disease pairs on one gated lead variant | 35,997 | 7,222 = 20.1% |
| disease pairs whose two terms share no therapeutic area | 19,147 | 5,099 = **26.6%** |
| **immune term against infection term — the axis** | **97** | **24 = 24.7%** |
| both terms immune-area | 2,010 | 266 = 13.2% |
| both terms infection-area | 42 | 5 = 11.9% |

**The axis is not elevated against a cross-area baseline on its own unit: 24.7% against 26.6%,
OR 0.91, P = 0.73.** Per-pair discordance rises with the variant's gated lead_vPS — 23.1% at 2
diseases through to 29.9% at 20 or more — and the lead variants carrying an axis disease pair are
much more pleiotropic than average, median gated lead_vPS 5.0 and mean 10.8 against 2.0 and 2.9 over
all lead variants with at least two gated diseases. Reweighting the cross-area baseline to the axis
pairs' lead_vPS bins gives **25.9%**, still above the axis's 24.7%.

The per-variant figure for the 34 lead variants carrying a gated pair, 22 of 34 = 64.7%, is that same
pleiotropy artefact and should not be quoted: a variant with 20 diseases is nearly certain to carry
one opposing pair whatever axis it sits on.

**What excess there is, is COVID-19.** Removing it takes the axis from 34.4% to **20.0%** per pair
and from 24.7% to **15.5%** per disease pair. Removing the two ontology caveats instead moves nothing
(34.4% to 37.5%, 24.7% to 22.7%). Removing all three leaves 13.3% and 9.7% — below the genome-wide
baseline on both units.

**So the letter should not claim this axis carries more antagonism than the genome as a whole.** The
matched comparison does not support it, and it is a claim the referee could check. The defensible
version is narrower and still answers the comment: allele-exact antagonism is present at this axis —
11 discordant gene x lead-variant pairs, named loci among them — at about the rate any
cross-therapeutic-area disease pair shows, which is roughly a quarter rather than the 7.5% the
published sentence implied. State the COVID-19 dependence in the same breath.

## The named examples all survive with their direction intact

| example | gated lead_vPS | concordance | immune side | infection side | verdict |
| --- | --- | --- | --- | --- | --- |
| IL23R / `C1orf141` `1_67131436_G_A` | 2 | **0.5000** (was 0.50) | Crohn's disease $-0.2739$ | leprosy $+0.6970$ | discordant |
| CCDC88B `11_64340263_G_A` | 8 | **0.6250** (was 0.667) | psoriasis $-0.0697$, autoimmune disease $-0.0395$, immune system disease $-0.1289$ | leprosy $+0.3774$ | discordant |
| TYK2 `19_10355447_C_T` | 2 | **0.5000** — confirmed | psoriatic arthritis $-0.1724$ | COVID-19 $+0.1559$ | discordant |
| LACC1 `13_43883789_A_G` | 2 | **1.0000** | Crohn's disease $+0.1384$ | leprosy $+1.1099$ | concordant |

Two flags:

- **CCDC88B's concordance moves, 0.667 to 0.6250.** Its gated lead_vPS is 8, not 3: sclerosing
  cholangitis, sarcoidosis, basal cell carcinoma and keratinocyte carcinoma carry neither area but do
  count toward the variant's concordance, and 2 of its 14 credible sets are dropped as unsigned. The
  immune/infection verdict is untouched.
- **IL23R, TYK2 `19_10355447_C_T` and LACC1 each rest on one gated credible set per side.** LACC1,
  the letter's concordant counterexample, is one Crohn's credible set against one leprosy credible
  set. All three should be described as single-study on both halves.

## TYK2 P1104A no longer forms a pair, and its counter-signal is on the wrong side

`19_10352442_G_C` has **no gated infection side**: its 3 COVID-19 credible sets are single-disease
but carry no signed effect, so the pair disappears. 15 of its 54 credible sets are dropped as
unsigned and 2 as multi-term, leaving gated lead_vPS 16 and concordance **0.9375**, still driven by
one positive effect. That effect is **tonsillitis $+0.1905$**, and `MONDO_0001039` carries
`EFO_0000540` (immune system disease) and **not** `EFO_0005741` — so under the class definition used
here and on 2026-08-18, tonsillitis sits on the **immune** side. The 2026-08-18 write-up presents it
as the infection counter-signal on clinical grounds and says so explicitly; that is defensible prose
but it is not a direction this machinery produces, and it must not be cited as one.

## The caveats after gating

- **COVID-19: 9 of 48 gated infection-side (lead variant, disease) rows, 18.8%, on 9 lead variants**,
  down from 49 comparable credible sets on 16 lead variants. The gate dilutes it, but it stays the
  largest single term and carries the whole excess over baseline.
- **The lymphoma terms still carry the infectious-disease area** through viral aetiology: Hodgkins
  lymphoma on 3 lead variants and extranodal nasal NK/T cell lymphoma on 1, 4 of 48 rows. They are
  cancers.
- **Peritonsillar abscess is a tie-break, and it is 7 of 48 rows.** `EFO_0007429` is the only corpus
  term carrying both areas, assigned to infection by the rule that infection wins. A seventh of the
  infection side therefore rests on that rule, and tonsillitis and its own suppurative complication
  end up on opposite sides of the axis.
- **No trans-disease study can supply both sides of a pair any more** — 0, and true by construction
  rather than by exclusion, since such a study carries at least two disease terms. The 81 credible
  sets reaching both areas at once, and the 122-to-59 halving the 2026-08-18 write-up reports, are
  artefacts of the old rule and no longer need reporting.